# Week 6 - Model Training, Tuning and Comparison

## Objective

Tune the Week 5 model frameworks using expanding-window time-series cross-validation with a 20-day gap, select models on the validation period, evaluate the historical test period, compare both ML approaches with the Week 4 BSM baseline, save trained estimators, and explain the selected models with SHAP.

Revision: Approach 2 now learns a bounded normalized time-value fraction with regularization and constrained moneyness tails. The old unbounded regression failed on live out-of-range input. The historical test was already viewed before this revision; its new metrics are retrospective, not an untouched holdout claim. Week 1–5 inputs and Approach 1 are retained.

## 1. Load Week 5 Dataset and Project Modules

In [1]:
from pathlib import Path
import json
import pickle
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
import shap
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=UserWarning)


def locate_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path(r"G:\JPM-Chooser Option Pricing")]
    for candidate in candidates:
        if (candidate / "Week 5" / "model_results" / "ml_dataset.csv").exists() and (candidate / "Week 4" / "bsm_chooser.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate Week 4 and Week 5 deliverables.")


PROJECT_ROOT = locate_project_root()
WEEK_DIR = Path.cwd()
RESULTS_DIR = WEEK_DIR / "model_results"
MODELS_DIR = WEEK_DIR / "trained_models"
FIGURES_DIR = WEEK_DIR / "figures"
for directory in [RESULTS_DIR, MODELS_DIR, FIGURES_DIR]:
    directory.mkdir(exist_ok=True)

sys.path.insert(0, str(WEEK_DIR))
sys.path.insert(0, str(PROJECT_ROOT / "Week 4"))
sys.path.insert(0, str(PROJECT_ROOT / "Week 5"))
from bsm_chooser import simple_chooser_price
from ml_features import PRICING_FEATURES, VOLATILITY_FEATURES
from model_training import metrics, purged_oof_predictions, refit_selected, tune_and_validate

with open(WEEK_DIR / "training_config.json", encoding="utf-8") as stream:
    CONFIG = json.load(stream)

dataset = pd.read_csv(PROJECT_ROOT / "Week 5" / "model_results" / "ml_dataset.csv", parse_dates=["Date"])
train = dataset[dataset["Split"] == "train"].copy()
validation = dataset[dataset["Split"] == "validation"].copy()
test = dataset[dataset["Split"] == "test"].copy()
print({part: len(frame) for part, frame in [("train", train), ("validation", validation), ("test", test)]})

{'train': 1156, 'validation': 232, 'test': 252}


## 2. Tune Approach 1: Volatility Forecasting

In [2]:
vol_models, vol_tuning = tune_and_validate(
    train[VOLATILITY_FEATURES], train["Target_Forward_Volatility_20D"],
    validation[VOLATILITY_FEATURES], validation["Target_Forward_Volatility_20D"],
    CONFIG["random_state"], "Forward volatility",
    CONFIG["time_series_cv_splits"], CONFIG["purge_gap_trading_days"],
)
selected_vol_name = vol_tuning.iloc[0]["Model"]
combined = pd.concat([train, validation], ignore_index=True)
best_vol_model = refit_selected(
    vol_models[selected_vol_name], combined[VOLATILITY_FEATURES], combined["Target_Forward_Volatility_20D"]
)
print(f"Selected Approach 1 model: {selected_vol_name}")
vol_tuning

Selected Approach 1 model: Random Forest
Out[2]: 
               Target  ...                                    Best_Parameters
0  Forward volatility  ...  {"model__max_depth": 8, "model__min_samples_le...
1  Forward volatility  ...  {"model__learning_rate": 0.03, "model__max_dep...
2  Forward volatility  ...                                                 {}
3  Forward volatility  ...                                                 {}

[4 rows x 8 columns]


## 3. Tune Approach 2: Direct Supervised Proxy Pricing

The direct-price estimator learns logit((proxy price - lower bound)/(upper bound - lower bound)). It uses no forecast-volatility target or BSM call at inference. Its moneyness tail coefficients are non-positive. Purged CV chooses regularization and raw versus volatility-scaled distance. Price-loss refinement was examined in development but not adopted; no revised design was chosen using the historical test target. These pointwise bounds do not establish full-surface arbitrage freedom.


In [3]:
price_models, price_tuning = tune_and_validate(
    train[PRICING_FEATURES], train["Target_Chooser_Proxy_Price"],
    validation[PRICING_FEATURES], validation["Target_Chooser_Proxy_Price"],
    CONFIG["random_state"], "Direct chooser proxy price",
    CONFIG["time_series_cv_splits"], CONFIG["purge_gap_trading_days"],
)
selected_price_name = price_tuning.iloc[0]["Model"]
best_price_model = refit_selected(
    price_models[selected_price_name], combined[PRICING_FEATURES], combined["Target_Chooser_Proxy_Price"]
)
print(f"Selected Approach 2 model: {selected_price_name}")
price_tuning

Selected Approach 2 model: Bounded time-value regression
Out[3]: 
                       Target  ...                                 Best_Parameters
0  Direct chooser proxy price  ...  {"alpha": 0.1, "distance_scale": "historical"}

[1 rows x 8 columns]


## 4. Purged Out-of-Fold Calibration Predictions

In [4]:
oof_volatility = purged_oof_predictions(
    vol_models[selected_vol_name], train[VOLATILITY_FEATURES], train["Target_Forward_Volatility_20D"],
    CONFIG["time_series_cv_splits"], CONFIG["purge_gap_trading_days"],
)
oof_mask = np.isfinite(oof_volatility)
calibration_output = train.loc[oof_mask, ["Date", "Close", "Treasury_Rate_Decimal", "Target_Chooser_Proxy_Price"]].copy()
calibration_output["Predicted_Forward_Volatility"] = np.clip(oof_volatility[oof_mask], 0.01, 2.0)
calibration_output["Approach1_Price"] = simple_chooser_price(
    calibration_output["Close"], 150.0, calibration_output["Treasury_Rate_Decimal"], 0.0233,
    calibration_output["Predicted_Forward_Volatility"], 0.5, 1.0,
)
calibration_output["Residual"] = calibration_output["Target_Chooser_Proxy_Price"] - calibration_output["Approach1_Price"]
assert len(calibration_output) > 0
assert calibration_output["Date"].max() < validation["Date"].min()
calibration_output.tail()

Out[4]: 
           Date       Close  ...  Approach1_Price  Residual
1151 2022-10-24  112.068626  ...        39.199607 -3.572361
1152 2022-10-25  112.361664  ...        38.628342 -2.959129
1153 2022-10-26  113.652863  ...        37.458127 -2.774455
1154 2022-10-27  114.101578  ...        37.154306 -2.723383
1155 2022-10-28  115.456879  ...        34.093617 -0.358408

[5 rows x 7 columns]


## 5. One-Time Test-Set Evaluation

In [5]:
contract = {"K": 150.0, "q": 0.0233, "T1": 0.5, "T2": 1.0}
test_output = test[["Date", "Close", "VIX_Close", "Target_Forward_Volatility_20D", "Target_Chooser_Proxy_Price", "Current_Chooser_BSM_Price"]].copy()

test_output["Persistence_Volatility"] = test["Rolling_Volatility_20D"]
test_output["ML_Predicted_Volatility"] = np.clip(best_vol_model.predict(test[VOLATILITY_FEATURES]), 0.01, 2.0)
test_output["Approach1_MLVol_BSM_Price"] = simple_chooser_price(
    test["Close"], contract["K"], test["Treasury_Rate_Decimal"], contract["q"],
    test_output["ML_Predicted_Volatility"], contract["T1"], contract["T2"]
)
test_output["Approach2_Direct_Price"] = best_price_model.predict(test[PRICING_FEATURES])

comparison_rows = [
    {"Approach": "Week 4 BSM baseline", "Model": "Trailing 20-day volatility", **metrics(test_output["Target_Chooser_Proxy_Price"], test_output["Current_Chooser_BSM_Price"])},
    {"Approach": "Approach 1", "Model": selected_vol_name + " volatility + BSM", **metrics(test_output["Target_Chooser_Proxy_Price"], test_output["Approach1_MLVol_BSM_Price"])},
    {"Approach": "Approach 2", "Model": selected_price_name + " direct proxy pricing", **metrics(test_output["Target_Chooser_Proxy_Price"], test_output["Approach2_Direct_Price"])},
]
model_comparison = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)

volatility_comparison = pd.DataFrame([
    {"Approach": "Persistence baseline", **metrics(test_output["Target_Forward_Volatility_20D"], test_output["Persistence_Volatility"])},
    {"Approach": "Selected ML volatility model", **metrics(test_output["Target_Forward_Volatility_20D"], test_output["ML_Predicted_Volatility"])},
])

model_comparison

Out[5]: 
              Approach  ... Mean_Error
0           Approach 1  ...  -1.278128
1           Approach 2  ...  -2.368891
2  Week 4 BSM baseline  ...  -0.529904

[3 rows x 7 columns]


## 6. Error Analysis by VIX Regime

In [6]:
test_output["VIX_Regime"] = pd.cut(test_output["VIX_Close"], [-np.inf, 15, 25, np.inf], labels=["Low (<15)", "Normal (15-25)", "High (>25)"], right=False)
regime_rows = []
for regime, group in test_output.groupby("VIX_Regime", observed=True):
    for label, column in [
        ("Week 4 BSM baseline", "Current_Chooser_BSM_Price"),
        ("Approach 1", "Approach1_MLVol_BSM_Price"),
        ("Approach 2", "Approach2_Direct_Price"),
    ]:
        regime_rows.append({"VIX_Regime": str(regime), "Approach": label, **metrics(group["Target_Chooser_Proxy_Price"], group[column])})
regime_metrics = pd.DataFrame(regime_rows)
regime_metrics

Out[6]: 
       VIX_Regime             Approach    N  ...       RMSE         R2  Mean_Error
0       Low (<15)  Week 4 BSM baseline  149  ...   3.649798   0.963848   -0.585975
1       Low (<15)           Approach 1  149  ...   3.341057   0.969706   -0.426435
2       Low (<15)           Approach 2  149  ...   3.044831   0.974839   -2.102599
3  Normal (15-25)  Week 4 BSM baseline  100  ...   6.214557   0.862913   -0.660597
4  Normal (15-25)           Approach 1  100  ...   5.304686   0.900116   -2.797745
5  Normal (15-25)           Approach 2  100  ...   5.364009   0.897870   -3.219806
6      High (>25)  Week 4 BSM baseline    3  ...   6.611795  -7.121340    6.611445
7      High (>25)           Approach 1    3  ...   8.688228 -13.023334    7.075000
8      High (>25)           Approach 2    3  ...  17.667586 -56.988690   12.769153

[9 rows x 7 columns]


## 7. Permutation Importance for Selected Models

In [7]:
importance_rows = []
for approach, model, features, target in [
    ("Approach 1", best_vol_model, VOLATILITY_FEATURES, "Target_Forward_Volatility_20D"),
    ("Approach 2", best_price_model, PRICING_FEATURES, "Target_Chooser_Proxy_Price"),
]:
    result = permutation_importance(
        model, test[features], test[target], n_repeats=15,
        random_state=CONFIG["random_state"], scoring="neg_mean_squared_error",
    )
    for feature, mean_value, std_value in zip(features, result.importances_mean, result.importances_std):
        importance_rows.append({"Approach": approach, "Feature": feature, "Importance_Mean": mean_value, "Importance_Std": std_value})
permutation_importance_table = pd.DataFrame(importance_rows).sort_values(["Approach", "Importance_Mean"], ascending=[True, False])
permutation_importance_table.groupby("Approach").head(10)

Out[7]: 
      Approach                  Feature  Importance_Mean  Importance_Std
6   Approach 1                VIX_Close         0.002267        0.000568
13  Approach 1                 MA50_Gap         0.000358        0.000080
9   Approach 1    Treasury_Rate_Decimal         0.000121        0.000044
8   Approach 1  VIX_JPM_Correlation_20D         0.000049        0.000168
2   Approach 1    Rolling_Volatility_5D         0.000031        0.000061
12  Approach 1                 MA20_Gap         0.000026        0.000007
3   Approach 1   Rolling_Volatility_10D         0.000021        0.000012
1   Approach 1            Abs_Return_1D         0.000002        0.000004
0   Approach 1             Daily_Return         0.000001        0.000002
10  Approach 1   Interest_Rate_Momentum         0.000001        0.000002
34  Approach 2                    Close       794.577979       47.369680
23  Approach 2                VIX_Close         3.170106        8.819469
25  Approach 2  VIX_JPM_Correlation_20D   

## 8. SHAP Analysis for the Selected Models

In [8]:
def selected_model_shap_table(pipeline, X, features, approach, target_name):
    if hasattr(pipeline, 'price_bounds'):
        # Explain the complete bounded PRICE function, not latent logit units.
        # Background comes only from development data. Fixed seed makes this
        # permutation-SHAP approximation reproducible.
        background = train[features].sample(min(40, len(train)), random_state=CONFIG['random_state'])
        def predict_array(a):
            return pipeline.predict(pd.DataFrame(a, columns=features))
        explainer = shap.Explainer(predict_array, background.to_numpy(), algorithm='permutation', seed=CONFIG['random_state'])
        explanation = explainer(X.to_numpy(), max_evals=2*len(features)+1)
        values = explanation.values
        table = pd.DataFrame({'Approach': approach, 'Target': target_name,
            'Feature': features, 'Mean_Absolute_SHAP': np.abs(values).mean(axis=0)})
        return X.to_numpy(), values, table.sort_values('Mean_Absolute_SHAP', ascending=False)
    transformed = X
    for _, transformer in pipeline.steps[:-1]:
        transformed = transformer.transform(transformed)
    model = pipeline.named_steps["model"]
    background = transformed[:min(100, len(transformed))]
    if hasattr(model, "estimators_") or model.__class__.__name__.startswith("XGB"):
        values = shap.TreeExplainer(model).shap_values(transformed)
    elif hasattr(model, "coef_"):
        values = shap.LinearExplainer(model, background).shap_values(transformed)
    else:
        values = shap.Explainer(model.predict, background)(transformed).values
    table = pd.DataFrame({
        "Approach": approach,
        "Target": target_name,
        "Feature": features,
        "Mean_Absolute_SHAP": np.abs(values).mean(axis=0),
    }).sort_values("Mean_Absolute_SHAP", ascending=False)
    return transformed, values, table


sample = test.tail(min(CONFIG["shap_sample_size"], len(test)))
vol_X, vol_shap, vol_shap_table = selected_model_shap_table(best_vol_model, sample[VOLATILITY_FEATURES], VOLATILITY_FEATURES, f"Approach 1 {selected_vol_name}", "Forward volatility")
price_X, price_shap, price_shap_table = selected_model_shap_table(best_price_model, sample[PRICING_FEATURES], PRICING_FEATURES, f"Approach 2 {selected_price_name}", "Direct chooser proxy price")
shap_importance = pd.concat([vol_shap_table, price_shap_table], ignore_index=True)
shap_importance.groupby("Approach").head(10)

Out[8]: 
                                    Approach  ... Mean_Absolute_SHAP
0                   Approach 1 Random Forest  ...           0.051947
1                   Approach 1 Random Forest  ...           0.033131
2                   Approach 1 Random Forest  ...           0.028022
3                   Approach 1 Random Forest  ...           0.008115
4                   Approach 1 Random Forest  ...           0.005818
5                   Approach 1 Random Forest  ...           0.005417
6                   Approach 1 Random Forest  ...           0.004035
7                   Approach 1 Random Forest  ...           0.001487
8                   Approach 1 Random Forest  ...           0.000963
9                   Approach 1 Random Forest  ...           0.000858
17  Approach 2 Bounded time-value regression  ...          13.798254
18  Approach 2 Bounded time-value regression  ...           5.745793
19  Approach 2 Bounded time-value regression  ...           2.138892
20  Approach 2 Bounded ti

## 9. Save Models, Tables and Visualizations

In [9]:
tuning_results = pd.concat([vol_tuning, price_tuning], ignore_index=True)
tuning_results.to_csv(RESULTS_DIR / "tuning_results.csv", index=False)
model_comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
volatility_comparison.to_csv(RESULTS_DIR / "volatility_comparison.csv", index=False)
test_output.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
regime_metrics.to_csv(RESULTS_DIR / "test_regime_metrics.csv", index=False)
permutation_importance_table.to_csv(RESULTS_DIR / "permutation_importance.csv", index=False)
shap_importance.to_csv(RESULTS_DIR / "shap_feature_importance.csv", index=False)
calibration_output.to_csv(RESULTS_DIR / "interval_calibration_predictions.csv", index=False)

# Keep joblib artifacts for the Week 8 application and export native pickle
# files to satisfy the Week 6 trained-model deliverable explicitly.
joblib.dump(best_vol_model, MODELS_DIR / "best_volatility_model.joblib")
joblib.dump(best_price_model, MODELS_DIR / "best_direct_pricing_model.joblib")

with open(MODELS_DIR / "best_volatility_model.pkl", "wb") as stream:
    pickle.dump(best_vol_model, stream, protocol=pickle.HIGHEST_PROTOCOL)
with open(MODELS_DIR / "best_direct_pricing_model.pkl", "wb") as stream:
    pickle.dump(best_price_model, stream, protocol=pickle.HIGHEST_PROTOCOL)

# Reload both pickle files and reproduce the complete held-out inference path.
with open(MODELS_DIR / "best_volatility_model.pkl", "rb") as stream:
    reloaded_vol_model = pickle.load(stream)
with open(MODELS_DIR / "best_direct_pricing_model.pkl", "rb") as stream:
    reloaded_price_model = pickle.load(stream)

verification_rows = test
reloaded_vol_predictions = np.clip(
    reloaded_vol_model.predict(verification_rows[VOLATILITY_FEATURES]), 0.01, 2.0
)
reloaded_price_predictions = reloaded_price_model.predict(verification_rows[PRICING_FEATURES])
assert np.allclose(
    test_output["ML_Predicted_Volatility"],
    reloaded_vol_predictions,
    rtol=0.0,
    atol=1e-12,
)
assert np.allclose(
    test_output["Approach2_Direct_Price"],
    reloaded_price_predictions,
    rtol=0.0,
    atol=1e-12,
)

metadata = {
    "direct_model_revision": "bounded_time_value_v1",
    "direct_model_module": "bounded_pricing.py",
    "direct_price_constraints": "abs(S exp(-qT)-K exp(-rT)) <= price <= S exp(-qT)+K exp(-rT)",
    "direct_model_parameters": best_price_model.get_params(),
    "revision_evaluation_policy": "Model revised after a live extrapolation failure. Hyperparameters selected by purged training CV. Historical test is reused for retrospective comparison, not a new untouched test.",
    "selected_volatility_model": selected_vol_name,
    "selected_direct_pricing_model": selected_price_name,
    "volatility_features": VOLATILITY_FEATURES,
    "pricing_features": PRICING_FEATURES,
    "contract": contract,
    "test_start": str(test["Date"].min().date()),
    "test_end": str(test["Date"].max().date()),
    "fit_start": str(combined["Date"].min().date()),
    "fit_end": str(combined["Date"].max().date()),
    "purge_gap_trading_days": CONFIG["purge_gap_trading_days"],
    "target_note": "Chooser target is an ex-post forward-volatility BSM proxy, not an observed chooser transaction price.",
}
pickle_validation = {
    "best_volatility_model": {
        "pickle_file": "best_volatility_model.pkl",
        "rows_checked": len(verification_rows),
        "max_absolute_difference_vs_saved_week6_predictions": float(
            np.max(np.abs(reloaded_vol_predictions - test_output["ML_Predicted_Volatility"]))
        ),
        "status": "passed",
    },
    "best_direct_pricing_model": {
        "pickle_file": "best_direct_pricing_model.pkl",
        "rows_checked": len(verification_rows),
        "max_absolute_difference_vs_saved_week6_predictions": float(
            np.max(np.abs(reloaded_price_predictions - test_output["Approach2_Direct_Price"]))
        ),
        "status": "passed",
    },
}
metadata["serialized_model_artifacts"] = [
    {"file": "best_volatility_model.pkl", "format": "pickle", "purpose": "Week 6 rubric deliverable"},
    {"file": "best_direct_pricing_model.pkl", "format": "pickle", "purpose": "Week 6 rubric deliverable"},
    {"file": "best_volatility_model.joblib", "format": "joblib", "purpose": "Week 8 runtime"},
    {"file": "best_direct_pricing_model.joblib", "format": "joblib", "purpose": "Week 8 runtime"},
]
metadata["pickle_round_trip_validation"] = pickle_validation
metadata["pickle_serialization_environment"] = {
    "python": sys.version.split()[0],
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
}
(MODELS_DIR / "model_bundle_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(test_output["Date"], test_output["Target_Chooser_Proxy_Price"], label="Proxy reference", color="black", linewidth=1.5)
ax.plot(test_output["Date"], test_output["Current_Chooser_BSM_Price"], label="Week 4 BSM", alpha=0.8)
ax.plot(test_output["Date"], test_output["Approach1_MLVol_BSM_Price"], label="Approach 1", alpha=0.8)
ax.plot(test_output["Date"], test_output["Approach2_Direct_Price"], label="Approach 2", alpha=0.8)
ax.set(title="Chooser Proxy Price Comparison on the Test Period", xlabel="Date", ylabel="Price ($)")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "test_price_comparison.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.8))
ordered = model_comparison.sort_values("RMSE")
ax.barh(ordered["Approach"], ordered["RMSE"], color=["#70ad47", "#5b9bd5", "#ed7d31"])
ax.set(title="Test RMSE by Pricing Approach", xlabel="RMSE ($)", ylabel="")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "test_rmse_comparison.png", dpi=180)
plt.close(fig)

for transformed, values, features, filename, title in [
    (vol_X, vol_shap, VOLATILITY_FEATURES, "approach1_shap_summary.png", f"Approach 1 {selected_vol_name} SHAP Summary"),
    (price_X, price_shap, PRICING_FEATURES, "approach2_shap_summary.png", f"Approach 2 {selected_price_name} SHAP Summary"),
]:
    shap.summary_plot(values, transformed, feature_names=features, max_display=12, show=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=180, bbox_inches="tight")
    plt.close()

print(f"Selected models: Approach 1 = {selected_vol_name}; Approach 2 = {selected_price_name}")
print(
    f"Saved {len(list(MODELS_DIR.glob('*.joblib')))} joblib files and "
    f"{len(list(MODELS_DIR.glob('*.pkl')))} pickle files; "
    "pickle round-trip prediction checks passed."
)
model_comparison

Selected models: Approach 1 = Random Forest; Approach 2 = Bounded time-value regression
Saved 2 joblib files and 2 pickle files; pickle round-trip prediction checks passed.
Out[9]: 
              Approach  ... Mean_Error
0           Approach 1  ...  -1.278128
1           Approach 2  ...  -2.368891
2  Week 4 BSM baseline  ...  -0.529904

[3 rows x 7 columns]


<ipython-input-9-b66b5821d4d4>:122: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(values, transformed, feature_names=features, max_display=12, show=False)
<ipython-input-9-b66b5821d4d4>:122: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(values, transformed, feature_names=features, max_display=12, show=False)


# Week 6 Conclusion

Hyperparameters were tuned with expanding-window cross-validation using a 20-day gap inside the training block. Candidate selection used the separate purged validation block, and revision metrics were recalculated on the previously observed historical test block. The comparison remains anchored to the Week 4 BSM baseline and Week 3 chooser formula. Permutation SHAP explains the complete bounded direct-price function in dollar units; Tree SHAP explains Approach 1 volatility, and the saved metadata records the feature lists, purge policy, contract, and proxy-target limitation.